# Distribuição dos produtos — Etapa 1 do plano de previsão

**Objetivo:** conhecer as séries produto × loja antes de prever qualquer coisa, e dividi-las em grupos que se comportam de forma parecida.

Cada grupo recebe um tratamento diferente nas próximas etapas:

| Etapa | Como usa os grupos deste notebook |
|---|---|
| 2 — Baselines (MM3, MM12, MM12 × índice sazonal) | Mede o erro de cada baseline **por grupo** e define quem recebe baseline e quem recebe regra própria (produtos novos e parados) |
| 3 — Engenharia de features | Os grupos viram features (classe ABC, frequência, fase de vida, alcance) e o índice sazonal escolhido aqui (geral ou por categoria) vira feature de calendário |
| 4 — Modelo | O resultado é comparado com a melhor baseline de cada grupo |

**Resposta curta:** as 6.054 séries formam uma grade completa (1.009 produtos × 6 locais), todas com pelo menos 3 anos de histórico, e se comportam de forma parecida. Não há produtos novos nem produtos descontinuados de verdade, o volume é espalhado, todas as categorias seguem a mesma estação e a demanda cresceu 15% no último ano. **Todas as séries entram juntas no mesmo modelo, com o índice sazonal geral e uma feature de tendência.**

**Em palavras simples:** este notebook é um "raio-x" do negócio antes de montar qualquer previsão. As perguntas são bem práticas: existe um produto que domina as vendas (e merece atenção especial)? Existem produtos "novos" que ainda não têm histórico suficiente? Existem produtos "mortos" que nunca mais vão vender? Cada categoria vende mais forte em épocas diferentes do ano? A resposta para quase tudo aqui é "não, é tudo bem parecido e regular" — o que é bom, porque significa que dá para tratar as 6.054 séries (produto × loja) da mesma forma, sem precisar de uma regra especial para cada tipo de produto.

**Como ler:** cada seção traz a pergunta, a decisão que depende dela, a célula de cálculo e um **Resultado** com o que os números mostram.

**Base:** `agg_demand_monthly`, só com meses completos. As janelas de 12 meses terminam no **mês de referência**: o último mês completo do dataset.

**Seções:**
1. Visão geral — quantas séries, produtos e lojas, e como o volume se divide
2. Concentração do volume (curva ABC)
3. Frequência de venda
4. Fase de vida (novas, ativas, paradas)
5. Sazonalidade por categoria
6. Tendência
7. Alcance dos produtos
8. Resumo: grupos que seguem para a Etapa 2

Os cálculos ficam em `src/analytics_ai/eda_demand_distribution.py`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Sobe a partir do diretório do notebook até achar a raiz do bundle (databricks.yml)."""
    for candidate in (start, *start.parents):
        if (candidate / "databricks.yml").exists():
            return candidate
    raise FileNotFoundError(f"databricks.yml não encontrado acima de {start}")


# Funciona no workspace (.bundle/.../files/notebooks/ml) e localmente.
PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from pyspark.sql import functions as F

from src.analytics_ai.eda_demand_distribution import (
    add_abc_class,
    add_frequency_class,
    add_life_phase,
    compare_to_global_season,
    complete_months,
    pareto_points,
    product_reach,
    reference_month,
    seasonal_index_by,
    segment_summary,
    series_profile,
    year_over_year_growth,
)
from src.analytics_ai.ml_config import MlFeatureConfig

# No Databricks o notebook já expõe `spark`; localmente cria via Databricks Connect
# (defina DATABRICKS_PROFILE com o nome do seu profile em ~/.databrickscfg).
spark = globals().get("spark")
if spark is None:
    from src.ingestion.utils import build_spark_session

    spark = build_spark_session()

# Databricks e Jupyter injetam `display`; fora deles, cai para print.
display = globals().get("display", print)

config = MlFeatureConfig.from_environment_monthly()
DATE_COL = config.date_col

monthly = spark.table(config.source_table)
products = spark.table(config.gold_table("dim_products")).select("sk_product", "category_name", "brand_name")
locations = spark.table(config.gold_table("dim_locations")).select(
    "sk_location", "location_name", "location_type", "region"
)
global_season = spark.table(config.silver_table("int_high_season_months"))

complete = complete_months(monthly, DATE_COL).join(products, "sk_product").join(locations, "sk_location")
REF_MONTH = reference_month(complete, DATE_COL)

print(f"tabela: {config.source_table}")
print(f"mês de referência (último mês completo): {REF_MONTH}")

### Perfil das séries

Uma linha por série produto × loja no mês de referência, com as três classificações usadas no resto do notebook:

| Coluna | O que é |
|---|---|
| `volume_12m`, `months_with_sale_12m` | Quantidade vendida e meses com venda nos últimos 12 meses |
| `history_months` | Meses desde a primeira venda |
| `months_since_last_sale` | Meses desde a última venda (0 = vendeu no mês de referência) |
| `abc_class` | **A** = séries que somam os primeiros 80% do volume; **B** = os 15% seguintes; **C** = os 5% finais; `sem_venda` = nada nos 12 meses |
| `frequency_class` | **alta** = 5 ou mais meses com venda; **media** = 3–4; **baixa** = 1–2; `sem_venda` = 0 |
| `life_phase` | **parada** = sem venda nos últimos 12 meses; **nova** = menos de 12 meses desde a primeira venda (a MM12 ainda não fecha); **ativa** = o resto |

O perfil tem ~6 mil linhas, então é trazido para memória uma vez e reaproveitado pelas seções seguintes.

In [ ]:
profile = series_profile(monthly, REF_MONTH, DATE_COL)
segmented_local = add_life_phase(add_frequency_class(add_abc_class(profile))).toPandas()
segmented = spark.createDataFrame(segmented_local).join(products, "sk_product").join(locations, "sk_location")

segmented_local.describe()

## 1. Visão geral

Tamanho do universo e como o volume dos últimos 12 meses se divide entre lojas.

**O que olhar:** se uma loja ou tipo de loja concentra o volume, o erro do modelo nela pesa mais no total e vale acompanhar separado.

In [ ]:
complete.agg(
    F.countDistinct("sk_product", "sk_location").alias("series"),
    F.countDistinct("sk_product").alias("produtos"),
    F.countDistinct("sk_location").alias("lojas"),
    F.countDistinct("category_name").alias("categorias"),
    F.min(DATE_COL).alias("primeiro_mes_completo"),
    F.max(DATE_COL).alias("ultimo_mes_completo"),
).toPandas()

In [ ]:
segment_summary(segmented, ["location_type", "location_name"]).toPandas()

**Resultado** *(16/09/2026, mês de referência: agosto/2026)*

- **6.054 séries = 1.009 produtos × 6 locais** (3 lojas e 3 armazéns), 11 categorias, meses completos de janeiro/2020 a agosto/2026.
- **Todo produto aparece em todos os locais.** Cada local tem 1.009 séries e responde por 16–17% do volume dos últimos 12 meses.

**O que significa:** nenhum local concentra o volume. Lojas e armazéns têm o mesmo peso, e o tipo de local deve ajudar pouco como feature.

## 2. Concentração do volume — curva ABC

**Pergunta:** o volume está concentrado em poucas séries ou espalhado?

- **Concentrado** (ex.: 20% das séries fazem 80% do volume): o erro nas séries A decide o resultado; vale avaliar o modelo ponderando por volume e olhar as séries A com lupa.
- **Espalhado** (precisa de metade das séries ou mais para chegar a 80%): não existe um grupo pequeno que resolva o problema; o modelo precisa funcionar bem para a maioria.

**Decisão que sai daqui:** a classe ABC vira feature (Etapa 3) e define o peso de cada série na avaliação (Etapa 2).

In [ ]:
pareto_points(segmented.filter(F.col("volume_12m") > 0)).toPandas()

In [ ]:
segment_summary(segmented, ["abc_class"]).toPandas()

**Resultado**

| Fatia das séries que mais vendem | Fatia do volume |
|---|---|
| 5% | 11,5% |
| 10% | 20,8% |
| 20% | 36,3% |
| 50% | 71,1% |

| Classe | Séries | Volume |
|---|---|---|
| A | 3.593 (59%) | 80% |
| B | 1.378 (23%) | 15% |
| C | 995 (16%) | 5% |
| Sem venda em 12 meses | 88 (1,5%) | 0% |

**O que significa:** o volume é **espalhado**. É preciso 59% das séries para chegar a 80% do volume (numa curva concentrada seriam ~20%). Não existe um grupo pequeno de séries que resolva o problema, então **não é preciso ponderar a avaliação por volume**: o modelo tem que funcionar para a maioria. A classe ABC entra como feature, mas deve separar pouco.

**Em palavras simples:** essa é a famosa "regra 80/20" (poucos produtos concentram a maior parte da venda) — aqui ela **não se aplica**. Numa loja onde a regra 80/20 valesse, bastariam os 20% de produtos "estrela" para somar 80% da venda. Aqui, é preciso quase 60% dos produtos para chegar nesses mesmos 80% — ou seja, a venda é bem repartida entre muitos produtos parecidos, sem "campeões de venda" isolados. Consequência prática: não adianta o modelo acertar bem só os produtos mais vendidos e errar o resto — ele precisa ser bom para o time inteiro, não só para os titulares.

## 3. Frequência de venda

**Pergunta:** em quantos dos últimos 12 meses cada série vendeu?

Isso diz quanto de informação cada média móvel tem para trabalhar. A MM3 olha só 3 meses: numa série que vende em poucos meses do ano, ela alterna entre zero e picos. A MM12 junta o ano inteiro e tende a ser mais estável. Quanto mais meses com venda, mais a MM3 tem chance de funcionar.

Os cortes das classes (**alta** = 5 ou mais meses com venda, **media** = 3–4, **baixa** = 1–2) seguem a distribuição destas séries, para cada classe ter volume suficiente para comparar.

O cruzamento com a classe ABC mostra se quem vende muito também vende com frequência, ou se o volume vem de pedidos grandes e raros.

**Decisão que sai daqui:** a frequência vira feature e é um dos cortes da comparação de baselines na Etapa 2.

In [ ]:
segment_summary(segmented, ["frequency_class"]).toPandas()

In [ ]:
abc_x_frequency = segment_summary(segmented, ["abc_class", "frequency_class"]).toPandas()

display(abc_x_frequency.pivot(index="abc_class", columns="frequency_class", values="n_series").fillna(0).astype(int))
abc_x_frequency.pivot(index="abc_class", columns="frequency_class", values="share_volume_12m").fillna(0)

**Resultado**

| Frequência | Meses com venda em 12 | Séries | Volume |
|---|---|---|---|
| Alta | 5 ou mais | 1.420 (23%) | 38% |
| Média | 3–4 | 2.939 (49%) | 49% |
| Baixa | 1–2 | 1.607 (27%) | 13% |
| Sem venda | 0 | 88 (1,5%) | 0% |

Cruzando com a ABC: a classe A é quase toda de frequência alta ou média; a classe C é quase toda de frequência baixa.

**O que significa:** uma série típica vende em ~3 dos 12 meses. Numa janela de 3 meses, isso é menos de 1 mês com venda, então **a MM3 vai alternar entre zero e picos na maioria das séries**. A expectativa é a MM12 se sair melhor, talvez com exceção da frequência alta. A Etapa 2 confirma quebrando o erro por essas três classes.

## 4. Fase de vida

**Pergunta:** quantas séries estão ativas, quantas são novas e quantas pararam de vender?

- **Paradas** (sem venda em 12 meses): candidatas a ficar fora do treino com previsão zero por regra. A regra só vale se elas **não voltarem a vender** — a seção testa isso olhando para trás.
- **Novas** (menos de 12 meses de histórico): a MM12 não fecha, então precisariam de uma regra própria.
- **Ativas:** o resto.

**Decisão que sai daqui:** se o modelo precisa de regras separadas para novas e paradas, ou se todas as séries entram juntas.

In [ ]:
segment_summary(segmented, ["life_phase"]).toPandas()

In [ ]:
# Meses desde a última venda, só entre as séries que ainda não estão paradas.
recency = segmented_local[segmented_local["life_phase"] != "parada"]["months_since_last_sale"]
recency_buckets = recency.apply(
    lambda m: "0 (vendeu no mês)" if m == 0 else "1–2" if m <= 2 else "3–5" if m <= 5 else "6–11"
)
recency_buckets.value_counts().reindex(["0 (vendeu no mês)", "1–2", "3–5", "6–11"]).to_frame("n_series")

### Séries paradas voltam a vender?

Olha para trás: toma as séries que estavam **paradas 12 meses antes do mês de referência** (sem venda nos 12 meses anteriores àquela data) e conta quantas venderam de novo nos 12 meses seguintes. É o teste direto da regra "parada = prever zero".

In [ ]:
# REF_MONTH é sempre o dia 1, então trocar o ano volta exatamente 12 meses.
past_ref = REF_MONTH.replace(year=REF_MONTH.year - 1)

past_profile = add_life_phase(series_profile(monthly, past_ref, DATE_COL))
stopped_then = past_profile.filter(F.col("life_phase") == "parada").select("sk_product", "sk_location")

came_back = stopped_then.join(segmented.select("sk_product", "sk_location", "volume_12m"), ["sk_product", "sk_location"])
came_back.agg(
    F.count("*").alias("paradas_12_meses_antes"),
    F.sum((F.col("volume_12m") > 0).cast("int")).alias("voltaram_a_vender"),
    F.round(F.mean((F.col("volume_12m") > 0).cast("int")), 4).alias("taxa_de_retorno"),
    F.sum("volume_12m").alias("volume_das_que_voltaram"),
).toPandas()

**Resultado**

| Fase | Séries |
|---|---|
| Ativa | 5.966 (98,5%) |
| Parada (sem venda em 12 meses) | 88 (1,5%) |
| Nova (menos de 12 meses de histórico) | **0** — a série mais recente tem 39 meses |

Meses desde a última venda entre as ativas: 959 venderam no próprio mês, 1.440 há 1–2 meses, 2.442 há 3–5 meses e 1.125 há 6–11 meses.

**Paradas voltam a vender?** Das 166 séries paradas 12 meses antes da referência, **163 (98%) voltaram a vender** nos 12 meses seguintes.

**O que significa:** "parada" aqui não é produto descontinuado: com uma venda a cada ~4 meses, passar 12 meses sem vender acontece por acaso. **Prever zero para as paradas seria errado.** E como não há séries novas, **nenhuma regra separada é necessária**: todas as 6.054 séries entram juntas no modelo, e "meses desde a última venda" vira feature.

## 5. Sazonalidade por categoria

**Pergunta:** todas as categorias seguem a estação geral (alta temporada de novembro a abril, pela regra de `int_high_season_months`)?

Para cada categoria, aplica a mesma regra dentro dela: demanda média de cada mês do ano ÷ média das 12 médias mensais da categoria.

**Como ler a comparação:**
- `correlation_with_global` perto de 1: a categoria tem o mesmo formato de estação que o geral.
- `max_abs_diff`: a maior distância entre o índice da categoria e o geral num mesmo mês.
- `months_differing_from_global`: em quantos meses a categoria discorda da classificação alta/baixa geral.

**Decisão que sai daqui:** se as categorias seguem o geral, basta o **índice sazonal geral** como feature e na baseline MM12 × índice. Se alguma categoria tiver estação própria (e volume suficiente para o índice ser confiável), usa-se o **índice por categoria**.

In [ ]:
category_season = seasonal_index_by(complete, ["category_name"], DATE_COL)

compare_to_global_season(category_season, global_season, "category_name").toPandas()

In [ ]:
category_season.toPandas().pivot(index="category_name", columns="month_number", values="seasonal_index").round(2)

In [ ]:
# Volume de cada categoria nos últimos 12 meses: categorias pequenas têm índice sazonal ruidoso.
segment_summary(segmented, ["category_name"]).toPandas().sort_values("volume_12m", ascending=False)

**Resultado**

- As 9 categorias com volume relevante (99,8% do total) têm estação praticamente igual à geral: **correlação de 0,985 a 0,995**, pico em janeiro e diferença máxima de índice de 0,12 a 0,25 num mesmo mês.
- Onde discordam da classificação geral, é quase sempre em **outubro ou abril**, os meses que já estão perto da média (0,99 e 1,05).
- **Pesca e Iluminação** (6 séries e ~150 unidades por ano cada) têm correlação menor (0,86–0,89), mas com esse volume o índice delas é ruído.

**O que significa:** **o índice sazonal geral basta.** Um índice por categoria não traria informação nova e seria ruidoso nas categorias pequenas.

**Em palavras simples:** "correlação" aqui mede o quanto o formato da estação de uma categoria parece com o formato da estação geral da loja, numa escala de −1 a 1 (1 = formato idêntico, 0 = nenhuma relação). Uma correlação de 0,985 a 0,995 é essencialmente "igual": se a categoria X vende mais em janeiro e menos em julho, isso acontece na mesma proporção para praticamente todas as categorias grandes. Só duas categorias pequenas (Pesca e Iluminação, com pouquíssimas vendas) mostram uma correlação mais baixa — mas isso é esperado: com tão poucos dados, qualquer índice calculado para elas seria pouco confiável, então a diferença não significa uma estação realmente diferente, e sim falta de dado suficiente para medir direito. Na prática: um único calendário sazonal serve para a loja inteira, não é preciso um por categoria.

## 6. Tendência

**Pergunta:** a demanda está crescendo, caindo ou estável?

Compara os últimos 12 meses com os 12 anteriores. Também conta quantas séries venderam em cada janela: crescimento que vem só de **mais séries vendendo** é diferente de **cada série vender mais**.

**Decisão que sai daqui:**
- Tendência forte: a MM12 fica atrasada (subestima em crescimento, superestima em queda) e a MM3 deve se sair melhor; a razão MM3 ÷ MM12 vira feature de tendência.
- Tendência fraca: a MM12 é uma boa referência de nível.

In [ ]:
year_over_year_growth(complete, REF_MONTH, date_col=DATE_COL).toPandas()

In [ ]:
year_over_year_growth(complete, REF_MONTH, ["category_name"], date_col=DATE_COL).toPandas().sort_values("growth")

In [ ]:
year_over_year_growth(complete, REF_MONTH, ["location_type", "location_name"], date_col=DATE_COL).toPandas()

**Resultado**

| Recorte | Últimos 12 meses × 12 anteriores | Séries vendendo |
|---|---|---|
| Total | **+15,2%** (120.988 → 139.423 un.) | 5.888 → 5.966 (+1,3%) |
| Categorias com volume relevante | +10% a +23% | estável |
| Locais | +10% a +24% | estável |

**O que significa:** a demanda cresce em todo lugar, e o crescimento vem de **cada série vender mais**, não de mais séries vendendo. Isso tem uma consequência direta: a MM12 olha para um período ~7 meses atrás do que está sendo previsto, e com 15% de crescimento ao ano **tende a subestimar a demanda em cerca de 9%**. A Etapa 2 mede esse viés, e a engenharia de features precisa de uma **feature de tendência**, calculada no nível da categoria ou do local, onde há volume para medir crescimento.

## 7. Alcance dos produtos

**Pergunta:** cada produto vende em quantas lojas?

Um produto vendido em várias lojas permite **emprestar sinal**: a venda dele nas outras lojas ajuda a prever cada loja. Um produto de uma loja só não tem esse apoio.

**Decisão que sai daqui:** o alcance vira feature, e a "demanda do produto em todas as lojas" entra no grupo de sinal emprestado da Etapa 3 se boa parte do volume vier de produtos vendidos em várias lojas.

In [ ]:
product_reach(segmented).toPandas()

**Resultado**

| Vendeu em quantos locais (12 meses) | Produtos | Volume |
|---|---|---|
| 6 | 924 (92%) | 93% |
| 5 | 82 (8%) | 7% |
| 4 | 3 (0,3%) | 0,2% |

**O que significa:** quase todo produto vende em todos os locais, então **a venda do mesmo produto nos outros locais está disponível para 93% do volume**. O notebook 03 testou essa informação: ela prevê tão bem quanto a média da categoria no local, e não melhor, porque o volume de um produto não se repete de um ano para o outro.

## 8. Resumo — grupos que seguem para a Etapa 2

Cruzamento final: fase de vida × classe ABC, com o volume de cada grupo. Mostra o universo que segue para a Etapa 2 e quais cortes usar para quebrar o erro das médias móveis.

In [ ]:
phase_x_abc = segment_summary(segmented, ["life_phase", "abc_class"]).toPandas()

display(phase_x_abc.pivot(index="life_phase", columns="abc_class", values="n_series").fillna(0).astype(int))
phase_x_abc.pivot(index="life_phase", columns="abc_class", values="share_volume_12m").fillna(0)

**Resultado**

Todas as séries com venda nos últimos 12 meses são ativas (5.966); as 88 paradas não têm classe ABC porque não venderam na janela.

**O que significa:** o universo da Etapa 2 são **todas as 6.054 séries**, avaliadas juntas e quebradas por **frequência de venda** (a divisão que mais diferencia as séries) e por categoria.

## Conclusão

**Em uma frase, para quem não trabalha com dados:** o negócio é bastante uniforme — nenhum produto "estrela", nenhum produto "moribundo", a mesma época de alta venda em todas as categorias — então dá para tratar todos os 6.054 produto × loja com a mesma régua, em vez de criar uma regra diferente para cada tipo de produto.

| Pergunta | Resposta | Decisão para as próximas etapas |
|---|---|---|
| O volume é concentrado? | Não: 59% das séries fazem 80% do volume | Avaliação sem ponderar por volume |
| Com que frequência as séries vendem? | ~3 meses em 12; 23% alta, 49% média, 27% baixa | Quebrar o erro por frequência; esperar MM3 ruidosa |
| Há séries novas ou descontinuadas? | Não: nenhuma nova, e 98% das "paradas" voltam a vender | Todas as séries no mesmo modelo, sem regras separadas; "meses desde a última venda" como feature |
| A estação muda por categoria? | Não: correlação ≥ 0,985 com a geral | Índice sazonal geral |
| Há tendência? | Sim: +15% ao ano, em todas as categorias e locais | Feature de tendência; medir o viés da MM12 na Etapa 2 |
| Dá para emprestar sinal entre locais? | Sim: 93% do volume vem de produtos vendidos nos 6 locais | Venda do produto nos outros locais como feature — **revisto no notebook 03**: o produto nos outros locais prevê tão bem quanto a categoria, e não melhor |

**Observação sobre os dados:** grade completa, locais com o mesmo volume, crescimento e estação iguais em todas as categorias. É um padrão muito regular, típico de dados simulados. Não muda as decisões, mas significa que o ganho de features por segmento (categoria, local) deve ser pequeno, e o que mais importa é tendência, estação e o sinal agregado.